# Chapter 11 &mdash; Completeness and Consistency of a Grammar

**Concept 6 of the Chapter 11 decomposition:** *Completeness and Consistency of a Grammar*

Complete = derives <i>all</i> intended strings; consistent = derives <i>nothing</i> unintended.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter11/Concept-Completeness-And-Consistency/Concept-Completeness-And-Consistency.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --

#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Two separate obligations, and a grammar can fail either one:

* **complete** &mdash; every string of the intended language is derivable
  ($L_{\text{intended}} \subseteq L(G)$);
* **consistent** (or *sound*) &mdash; nothing else is
  ($L(G) \subseteq L_{\text{intended}}$).

Testing them is asymmetric. Completeness is checked by **generating** and looking for
gaps; consistency by **parsing** things that should fail.

The proofs are two inductions: completeness by induction on the string, consistency by
induction on the derivation (equivalently, on the parse tree).

## 2. Definitions

### The CFG toolkit

A grammar is a dict; `language`, `nparses`, `parse_trees` and `leftmost` do the work.

In [ ]:
# --- a tiny CFG toolkit -------------------------------------------------
# A grammar is a dict with keys N (nonterminals), Sigma (terminals),
# S (start symbol) and P (productions: nonterminal -> list of RHS tuples).
# A right-hand side is a tuple of one-character symbols; () is epsilon.
# By convention UPPERCASE single letters are nonterminals.

def mkg(rules, start='S'):
    N = set(rules)
    P = {A: [tuple(r) for r in rhs] for A, rhs in rules.items()}
    Sigma = {c for rhs in P.values() for r in rhs for c in r if c not in N}
    return dict(N=N, Sigma=Sigma, S=start, P=P)

def show(G):
    print("N     =", sorted(G['N']))
    print("Sigma =", sorted(G['Sigma']))
    print("S     =", G['S'])
    for A in sorted(G['P']):
        alts = ' | '.join((''.join(r) if r else "''") for r in G['P'][A])
        print("   %s -> %s" % (A, alts))

def derivable(G, maxlen):
    # least fixed point: for each nonterminal, every terminal string of
    # length <= maxlen it derives.  Far cheaper than searching sentential
    # forms, and it terminates because the sets only grow and are bounded.
    T = {A: set() for A in G['N']}
    def spans(r):
        acc = {''}
        for x in r:
            src = T[x] if x in T else {x}
            acc = {a + b for a in acc for b in src if len(a) + len(b) <= maxlen}
            if not acc: break
        return acc
    changed = True
    while changed:
        changed = False
        for A in G['P']:
            for r in G['P'][A]:
                for w in spans(r):
                    if w not in T[A]:
                        T[A].add(w); changed = True
    return T

def language(G, maxlen):
    return sorted(derivable(G, maxlen)[G['S']], key=lambda s: (len(s), s))

def _spans(G, w, cap=None):
    # Bottom-up, shortest span first, so a span never depends on a LONGER
    # one.  Within a span we iterate |N|+1 times, which is enough to close
    # unit rules (A -> B) and epsilon rules.  Doing it top-down with a
    # "cycle guard" silently poisons the memo table, so we do not.
    n, N, P = len(w), G['N'], G['P']
    tab = {}                       # (A, i, j) -> count, or list of trees
    def get(sym, i, j):
        if sym not in N:
            if j == i + 1 and w[i] == sym:
                return 1 if cap is None else [sym]
            return 0 if cap is None else []
        return tab.get((sym, i, j), 0 if cap is None else [])
    def seqv(r, i, j):
        if not r:
            if i != j: return 0 if cap is None else []
            return 1 if cap is None else [()]
        acc = 0 if cap is None else []
        for k in range(i, j + 1):
            a = get(r[0], i, k)
            if not a: continue
            b = seqv(r[1:], k, j)
            if not b: continue
            if cap is None:
                acc += a * b
            else:
                for h in a:
                    for t in b:
                        acc.append((h,) + tuple(t))
                        if len(acc) >= cap: return acc
        return acc
    for length in range(0, n + 1):
        for i in range(0, n - length + 1):
            j = i + length
            for _ in range(len(N) + 1):
                grew = False
                for A in P:
                    v = []
                    for r in P[A]:
                        x = seqv(r, i, j)
                        if cap is None:
                            v.append(x)
                        else:
                            v += [(A,) + tuple(t) for t in x]
                            if len(v) >= cap: v = v[:cap]; break
                    v = sum(v) if cap is None else v
                    old = tab.get((A, i, j), 0 if cap is None else [])
                    if (v != old) if cap is None else (len(v) != len(old)):
                        tab[(A, i, j)] = v; grew = True
                if not grew: break
    return get(G['S'], 0, n)

def nparses(G, w):
    return _spans(G, w, cap=None)

def parse_trees(G, w, cap=8):
    return _spans(G, w, cap=cap)

def yield_of(t):
    return t if isinstance(t, str) else ''.join(yield_of(c) for c in t[1:])

def show_tree(t, ind=0):
    if isinstance(t, str):
        print("%s'%s'" % ('  ' * ind, t)); return
    print("%s%s" % ('  ' * ind, t[0]))
    for c in t[1:]: show_tree(c, ind + 1)

def leftmost(G, w):
    # the leftmost derivation read off one parse tree
    ts = parse_trees(G, w, cap=1)
    if not ts: return None
    steps, form = [], [G['S']]
    def expand(t, pos):
        # t is the subtree rooted at the nonterminal currently at `pos`
        if isinstance(t, str): return pos + 1
        kids = [c if isinstance(c, str) else c[0] for c in t[1:]]
        form[pos:pos+1] = kids
        steps.append(''.join(form) or "''")
        p = pos
        for c in t[1:]:
            p = expand(c, p)
        return p
    steps.append(G['S'])
    expand(ts[0], 0)
    return steps

### An intended language, and three candidate grammars

In [ ]:
def intended(s):
    # a^n b^n, n >= 0
    k = len(s) - len(s.lstrip('a'))
    return s == 'a' * k + 'b' * (len(s) - k) and k == len(s) - k

Good       = mkg({'S': ["", "aSb"]})
Incomplete = mkg({'S': ["ab", "aSb"]})           # misses epsilon
Inconsist  = mkg({'S': ["", "aSb", "ba"]})       # generates 'ba' too

### The two tests

In [ ]:
from itertools import product
def audit(G, upto=8, sigma='ab'):
    gen = set(language(G, upto))
    want = {''.join(p) for k in range(upto+1) for p in product(sigma, repeat=k)
            if intended(''.join(p))}
    return dict(missing=sorted(want - gen), extra=sorted(gen - want))

## 3. Tests

The good grammar passes both tests.

In [ ]:
a = audit(Good)
print("missing :", a['missing'])
print("extra   :", a['extra'])
assert not a['missing'] and not a['extra']
print("complete AND consistent, up to length 8")

The incomplete grammar has a **gap**.

In [ ]:
a = audit(Incomplete)
print("missing :", a['missing'], "  <- incomplete")
print("extra   :", a['extra'])
assert a['missing'] == [''] and not a['extra']
print("\nEvery string it DOES derive is intended -- consistency is not the problem.")

The inconsistent grammar derives **too much**.

In [ ]:
a = audit(Inconsist)
print("missing :", a['missing'])
print("extra   :", a['extra'], "  <- inconsistent")
assert not a['missing'] and 'ba' in a['extra']
print("\nCompleteness is fine; it is soundness that fails.")

The two obligations are genuinely independent.

In [ ]:
print("%-12s %-12s %-12s" % ("grammar", "complete", "consistent"))
for name, G in [('Good', Good), ('Incomplete', Incomplete), ('Inconsist', Inconsist)]:
    a = audit(G)
    print("%-12s %-12s %-12s" % (name, not a['missing'], not a['extra']))

The completeness induction, exhibited: every $a^nb^n$ has a derivation.

In [ ]:
for n in range(6):
    w = 'a'*n + 'b'*n
    d = leftmost(Good, w)
    assert d is not None and d[-1] == (w or "''")
    print("  n=%d : %s" % (n, ' => '.join(d)))

## 4. Exercises


1. Write the consistency induction for `S -> '' | aSb` in full.
2. Give a grammar that is complete and inconsistent for $L_{Dyck}$.
3. Why can neither test ever be *finished* by testing alone?

In [ ]:
# Your work for the exercises above.